## Import libraries

In [57]:
import io
import sys
import random
import json

import numpy as np
import pandas as pd


#_________
from tqdm.auto import tqdm

## Utility functions

### safe_unzip()

In [5]:
import zipfile
import os

def safe_unzip(zip_path, extract_path=None):
    try:
        # Validate zip file exists
        if not os.path.exists(zip_path):
            raise FileNotFoundError(f"Zip file not found: {zip_path}")
        
        # Set extract path
        if extract_path is None:
            extract_path = os.path.splitext(zip_path)[0]
        
        # Create directory if needed
        os.makedirs(extract_path, exist_ok=True)
        
        # Extract
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            # Test if zip is valid
            bad_file = zip_ref.testzip()
            if bad_file:
                raise zipfile.BadZipFile(f"Corrupted file in zip: {bad_file}")
            
            zip_ref.extractall(extract_path)
            print(f"Successfully extracted to: {extract_path}")
            
    except FileNotFoundError as e:
        print(f"Error: {e}")
    except zipfile.BadZipFile as e:
        print(f"Error: Invalid or corrupted zip file - {e}")
    except Exception as e:
        print(f"Unexpected error: {e}")

### read_text_file()

In [29]:
def read_text_file(filepath):

    '''
    Description
    -----------

        =================
        Phonetic info
        this is the structure of a .phn file 
            0 3050 h#
            3050 4559 sh
            4559 5723 ix
            5723 6642 hv
            6642 8772 eh
            8772 9190 dcl
            9190 10337 jh

            as you can see it is seperator by space, it looks like time points (for sr=16000)
                and the last [-1] column is the one with the Arphable phonetic symbole. 


        =================
        word info
        similar happens with .wrd file 
            3050 5723 she
            5723 10337 had
            9190 11517 your
            11517 16334 dark
            16334 21199 suit
            21199 22560 in
            22560 28064 greasy
            28064 33360 wash
            33754 37556 water
            37556 40313 all
            40313 44586 year
        as you can see it is seperator by space, it looks like time points (for sr=16000)
            and the last [-1] column per line is the one that contains the word. 
    
    '''

    
    with open(filepath) as f:
        tokens = [line.split()[-1] for line in f]
        return " ".join(tokens)

## Unzip

In [9]:
TIMIT_KAGGLE_ROOT = os.path.join("..", "..", 
                                 'data', 
                                 '07_timit_kaggle'
                                 
                                )
#=========================================================
data_path = os.path.join(TIMIT_KAGGLE_ROOT, 'data') 

In [10]:
%%time
zip_fname = 'timit_kaggle.zip'
zip_fpath = os.path.join(TIMIT_KAGGLE_ROOT, zip_fname)

zip_extract_path = TIMIT_KAGGLE_ROOT

safe_unzip(zip_path=zip_fpath, extract_path=zip_extract_path)
print(f"zip file has been extrated in this folder: \n {zip_extract_path}")

Successfully extracted to: ../../data/07_timit_kaggle
zip file has been extrated in this folder: 
 ../../data/07_timit_kaggle
CPU times: user 32.9 s, sys: 22.4 s, total: 55.3 s
Wall time: 29min 41s


## Load meta timit

In [11]:
df_train = pd.read_csv(os.path.join(TIMIT_KAGGLE_ROOT, 'train_data.csv'))
df_train.head(3)

,index,test_or_train,dialect_region,speaker_id,filename,path_from_data_dir,path_from_data_dir_windows,is_converted_audio,is_audio,is_word_file,is_phonetic_file,is_sentence_file
0,1.0,TRAIN,DR4,MMDM0,SI681.WAV.wav,TRAIN/DR4/MMDM0/SI681.WAV.wav,TRAIN\\DR4\\MMDM0\\SI681.WAV.wav,True,True,False,False,False
1,2.0,TRAIN,DR4,MMDM0,SI1311.PHN,TRAIN/DR4/MMDM0/SI1311.PHN,TRAIN\\DR4\\MMDM0\\SI1311.PHN,False,False,False,True,False
2,3.0,TRAIN,DR4,MMDM0,SI1311.WRD,TRAIN/DR4/MMDM0/SI1311.WRD,TRAIN\\DR4\\MMDM0\\SI1311.WRD,False,False,True,False,False


In [12]:
df_test = pd.read_csv(os.path.join(TIMIT_KAGGLE_ROOT, 'test_data.csv'))
df_test.head(3)

,index,test_or_train,dialect_region,speaker_id,filename,path_from_data_dir,path_from_data_dir_windows,is_converted_audio,is_audio,is_word_file,is_phonetic_file,is_sentence_file
0,1.0,TEST,DR4,MGMM0,SX139.WAV,TEST/DR4/MGMM0/SX139.WAV,TEST\\DR4\\MGMM0\\SX139.WAV,False,True,False,False,False
1,2.0,TEST,DR4,MGMM0,SX139.WAV.wav,TEST/DR4/MGMM0/SX139.WAV.wav,TEST\\DR4\\MGMM0\\SX139.WAV.wav,True,True,False,False,False
2,3.0,TEST,DR4,MGMM0,SX139.TXT,TEST/DR4/MGMM0/SX139.TXT,TEST\\DR4\\MGMM0\\SX139.TXT,False,False,False,False,True


In [13]:
df = pd.concat([df_train, df_test])
df.head(3)

,index,test_or_train,dialect_region,speaker_id,filename,path_from_data_dir,path_from_data_dir_windows,is_converted_audio,is_audio,is_word_file,is_phonetic_file,is_sentence_file
0,1.0,TRAIN,DR4,MMDM0,SI681.WAV.wav,TRAIN/DR4/MMDM0/SI681.WAV.wav,TRAIN\\DR4\\MMDM0\\SI681.WAV.wav,True,True,False,False,False
1,2.0,TRAIN,DR4,MMDM0,SI1311.PHN,TRAIN/DR4/MMDM0/SI1311.PHN,TRAIN\\DR4\\MMDM0\\SI1311.PHN,False,False,False,True,False
2,3.0,TRAIN,DR4,MMDM0,SI1311.WRD,TRAIN/DR4/MMDM0/SI1311.WRD,TRAIN\\DR4\\MMDM0\\SI1311.WRD,False,False,True,False,False


In [14]:
c1 = df['is_converted_audio'] == False
df = df[c1].reset_index(drop=True)
df.shape

(25200, 12)

In [17]:
%%time
data = {}

for idx, row in tqdm(df.iterrows()):

    # meta file with linux sys path sep to audio file. 
    #  Each audio file name is related to subID
    #  the root folder that a "data" folder with all wav file, phonetic, word files. 
    path = row['path_from_data_dir'] 

    # Extracting from path of wav the subId path. 
    entry_id = path.split('.')[0]

    #__________________________________
    # Creating empty dictionary to story subID info.
    if entry_id not in data:
        data[entry_id] = {}

    #__________________________________
    # data_path is the root in the system to "data" folder in TIMIT kaggle root folder.
    if row['is_audio'] is True:
        data[entry_id]['audio_file'] = os.path.join(data_path, path)

    elif row['is_word_file'] is True:
        data[entry_id]['word_file'] = os.path.join(data_path, path)

    elif row['is_phonetic_file'] is True:
        data[entry_id]['phonetic_file'] = os.path.join(data_path, path)


    #__________________________________
    
    # break

0it [00:00, ?it/s]

CPU times: user 1.34 s, sys: 25 ms, total: 1.36 s
Wall time: 2.49 s


In [19]:
len(data.keys()) # 

6300

### Extracting subID with 3 files

In [20]:
keys = [key for key in data.keys() if len(data[key]) == 3]
len(keys)

3360

### Split data into train,val,test

In [24]:
random.Random(101).shuffle(keys)

num_train = int(len(keys) * 0.8)
num_valid = int(len(keys) * 0.1)
num_test = len(keys) - num_train - num_valid

train_keys = keys[:num_train]
valid_keys = keys[num_train:num_train + num_valid]
test_keys = keys[-num_test:]

In [25]:
print(f"train_keys {len(train_keys )}")
print(f"valid_keys {len(valid_keys )}")
print(f"test_keys {len(test_keys )}")

train_keys 2688
valid_keys 336
test_keys 336


In [26]:
# form the data dictionary and with the keys split into train, val and test, organize data for each set 
train = { key:data[key] for key in train_keys }
valid = { key:data[key] for key in valid_keys }
test  = { key:data[key] for key in test_keys }

### SAVE the dictionary as json files 

In [27]:
custom_split_data_path = os.path.join(TIMIT_KAGGLE_ROOT, "01_custom_split_meta")

os.makedirs(custom_split_data_path, exist_ok=True)
custom_split_data_path

'../../data/07_timit_kaggle/01_custom_split_meta'

In [28]:
fnames = ['custom_train.json', 'custom_valid.json', 'custom_test.json']

for fname in fnames: 
    fpath = os.path.join(custom_split_data_path,fname )
    with open(fpath, "w") as f:
        json.dump(train, f)

## Extracting one file .phn

In [33]:
sub_ids_lst = list(train.keys())
sub_ids_lst[:5]

['TRAIN/DR3/MILB0/SI2163',
 'TEST/DR6/MESD0/SX372',
 'TRAIN/DR2/FDNC0/SX378',
 'TRAIN/DR4/FSSB0/SI2342',
 'TEST/DR5/MDWA0/SA2']

In [35]:
sID = sub_ids_lst[0]
sID

'TRAIN/DR3/MILB0/SI2163'

In [36]:
train[sID]

{'audio_file': '../../data/07_timit_kaggle/data/TRAIN/DR3/MILB0/SI2163.WAV',
 'phonetic_file': '../../data/07_timit_kaggle/data/TRAIN/DR3/MILB0/SI2163.PHN',
 'word_file': '../../data/07_timit_kaggle/data/TRAIN/DR3/MILB0/SI2163.WRD'}

In [67]:
# phones_seg = read_text_file(filepath=)
# phones_seg

In [68]:
fpath_phone = train[sID]['phonetic_file']
filepath = fpath_phone
with open(filepath) as f:
    tokens = [line.split() for line in f]
d_phone = pd.DataFrame(tokens, columns=["start", "end", "label"])
d_phone['start'] = d_phone['start'].astype(int)
d_phone['end'] = d_phone['end'].astype(int)
d_phone.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34 entries, 0 to 33
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   start   34 non-null     int64 
 1   end     34 non-null     int64 
 2   label   34 non-null     object
dtypes: int64(2), object(1)
memory usage: 944.0+ bytes


## Creating dummy pred .phn file 

In [70]:
## Creating dummy prediction
sr = 16000
range_width = sr*.025 # 25ms

d_phone_pred = d_phone.copy()

d_phone_pred["start_pred"] = d_phone_pred["start"].apply(
    lambda x: np.random.randint(x - range_width, x + range_width + 1)
)

d_phone_pred["end_pred"] = d_phone_pred["end"].apply(
    lambda x: np.random.randint(x - range_width, x + range_width + 1)
)

cols = ["start_pred", "end_pred", "label"]
d_phone_pred = d_phone_pred[cols].copy()
d_phone_pred

,start_pred,end_pred,label
0,-155,1962,h#
1,1667,2294,q
2,2501,2862,aa
3,3558,3424,r
4,3283,4524,y
5,4410,5104,ux
6,5144,6804,pcl
7,6089,7674,p
8,7372,9011,aa
9,9132,10484,z


## Sequence matcher 

In [71]:
import difflib

In [104]:
ground_truth = ['w', 'o','o', 'r', 'd', 's']
asr_output = ['w', 'o', 'r', 'd','s','z']

In [105]:
## class instantiation
matcher = difflib.SequenceMatcher(isjunk=None, 
                                  a=ground_truth, 
                                  b=asr_output,
                                 )

In [106]:
for tag, i1, i2, j1, j2 in matcher.get_opcodes():
    if tag == 'equal':
        for i in range(i2 - i1):
            print(f"✅ Correct: {ground_truth[i1 + i]}")
    elif tag == 'replace':
        for i in range(max(i2 - i1, j2 - j1)):
            g = ground_truth[i1 + i] if i1 + i < i2 else "-"
            a = asr_output[j1 + i] if j1 + i < j2 else "-"
            print(f"❌ Wrong: {g} → {a}")
    elif tag == 'delete':
        for i in range(i2 - i1):
            prev = ground_truth[i1 + i - 1] if i1 + i - 1 >= 0 else None
            nxt = ground_truth[i1 + i + 1] if i1 + i + 1 < len(ground_truth) else None
            print(f"🕳️ Missing: {ground_truth[i1 + i]} (between {prev} and {nxt})")
    elif tag == 'insert':
        for i in range(j2 - j1):
            print(f"➕ Extra: {asr_output[j1 + i]} (not in Expected Sound)")

✅ Correct: w
🕳️ Missing: o (between w and o)
✅ Correct: o
✅ Correct: r
✅ Correct: d
✅ Correct: s
➕ Extra: z (not in Expected Sound)
